# Experiment 6 — Part 2: Video Understanding Using CNN + RNN
### CS3807 Deep Learning Laboratory — UCF101 | MobileNetV2 (frozen) -> LSTM(32) / GRU(32)
**Additional Exercise #6:** compare LSTM and GRU on *identical* cached CNN features.

Pipeline: `Video -> 10 uniform frames (224x224x3 RGB) -> frozen MobileNetV2 (pooling='avg') -> 10xD sequence -> LSTM/GRU(32) -> Dense -> Softmax`

## 1. Imports, configuration, seeds

In [1]:
import os, sys, json, time, random, glob, warnings, gc
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ---------------- configuration ----------------
NUM_FRAMES   = 10        # uniformly sampled frames per video
IMG_SIZE     = 224
MAX_PER_CLASS= 40        # videos per class (balanced, kept small for compute)
TARGET_CLASSES = 5
BATCH_SIZE   = 16
LR           = 1e-3
MAX_EPOCHS   = 30
RNN_UNITS    = 32
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

OUT = "experiment6_part2_outputs"
DIRS = {k: os.path.join(OUT, k) for k in
        ["plots", "confusion_matrices", "features", "model_results", "tables"]}
for d in DIRS.values(): os.makedirs(d, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"white","savefig.facecolor":"white",
    "font.size":12,"axes.titlesize":15,"axes.labelsize":13,
    "xtick.labelsize":11,"ytick.labelsize":11,"legend.fontsize":11,
    "axes.grid":True,"grid.alpha":0.3,"axes.spines.top":False,"axes.spines.right":False})

def save_fig(fig, folder, name):
    p = os.path.join(DIRS[folder], name)
    fig.tight_layout(); fig.savefig(p, dpi=300, bbox_inches="tight"); plt.close(fig)
    print("saved:", p); return p

GENERATED_PNGS = []
print("TF", tf.__version__, "| OpenCV", cv2.__version__, "| output dir:", OUT)

TF 2.20.0 | OpenCV 4.13.0 | output dir: experiment6_part2_outputs


## 2. GPU check

In [2]:
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass
print("GPUs detected:", len(gpus), gpus if gpus else "(running on CPU)")

GPUs detected: 2 [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 3. Locate UCF101

In [4]:
import os, glob, shutil, subprocess, sys

# ============================================================
# AUTOMATIC UCF101 DOWNLOAD
# ============================================================

UCF_WORK = "/kaggle/working/ucf101_data"
os.makedirs(UCF_WORK, exist_ok=True)

VIDEO_EXT = (".avi", ".mp4", ".mkv", ".mov", ".mpg", ".mpeg", ".webm")

UCF101_CLASSES = set("""
ApplyEyeMakeup ApplyLipstick Archery BabyCrawling BalanceBeam BandMarching BaseballPitch
Basketball BasketballDunk BenchPress Biking Billiards BlowDryHair BlowingCandles BodyWeightSquats
Bowling BoxingPunchingBag BoxingSpeedBag BreastStroke BrushingTeeth CleanAndJerk CliffDiving
CricketBowling CricketShot CuttingInKitchen Diving Drumming Fencing FieldHockeyPenalty
FloorGymnastics FrisbeeCatch FrontCrawl GolfSwing Haircut Hammering HammerThrow HandstandPushups
HandstandWalking HeadMassage HighJump HorseRace HorseRiding HulaHoop IceDancing JavelinThrow
JugglingBalls JumpingJack JumpRope Kayaking Knitting LongJump Lunges MilitaryParade Mixing
MoppingFloor Nunchucks ParallelBars PizzaTossing PlayingCello PlayingDaf PlayingDhol PlayingFlute
PlayingGuitar PlayingPiano PlayingSitar PlayingTabla PlayingViolin PoleVault PommelHorse PullUps
Punch PushUps Rafting RockClimbingIndoor RopeClimbing Rowing SalsaSpin ShavingBeard Shotput
SkateBoarding Skiing Skijet SkyDiving SoccerJuggling SoccerPenalty StillRings SumoWrestling
Surfing Swing TableTennisShot TaiChi TennisSwing ThrowDiscus TrampolineJumping Typing UnevenBars
VolleyballSpiking WalkingWithDog WallPushups WritingOnBoard YoYo
""".split())

# ------------------------------------------------------------
# Check whether UCF101 is already available
# ------------------------------------------------------------

def find_ucf101(root):
    for dirpath, dirnames, filenames in os.walk(root):
        class_dirs = [d for d in dirnames if d in UCF101_CLASSES]

        if len(class_dirs) >= 3:
            idx = {}

            for cls in class_dirs:
                paths = []
                for ext in VIDEO_EXT:
                    paths.extend(
                        glob.glob(
                            os.path.join(dirpath, cls, f"*{ext}"),
                            recursive=False
                        )
                    )

                if paths:
                    idx[cls] = sorted(paths)

            if len(idx) >= 3:
                return idx, dirpath

    return {}, None


VIDEO_INDEX, UCF_ROOT = find_ucf101("/kaggle/input")

# ------------------------------------------------------------
# If not found, download original UCF101 from Kaggle
# ------------------------------------------------------------

if not VIDEO_INDEX:

    print("UCF101 not found in /kaggle/input.")
    print("Downloading UCF101 from Kaggle...")

    try:
        import kagglehub
    except ImportError:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "-q", "kagglehub"
        ])
        import kagglehub

    dataset_path = kagglehub.dataset_download("pevogam/ucf101")

    print("Kaggle dataset downloaded to:")
    print(dataset_path)

    # Search downloaded dataset
    VIDEO_INDEX, UCF_ROOT = find_ucf101(dataset_path)

# ------------------------------------------------------------
# Final check
# ------------------------------------------------------------

if not VIDEO_INDEX:
    raise FileNotFoundError(
        "UCF101 could not be located after automatic download. "
        "Check the Kaggle internet setting and dataset availability."
    )

print("\nUCF101 FOUND")
print("Root:", UCF_ROOT)
print("Classes found:", len(VIDEO_INDEX))
print("Total videos:", sum(len(v) for v in VIDEO_INDEX.values()))

print("\nExample classes:")
for cls in list(VIDEO_INDEX.keys())[:10]:
    print(f"{cls}: {len(VIDEO_INDEX[cls])} videos")

UCF101 not found in /kaggle/input.
Kaggle dataset downloaded to:
/kaggle/input/datasets/pevogam/ucf101

UCF101 FOUND
Root: /kaggle/input/datasets/pevogam/ucf101/UCF101/UCF-101
Classes found: 101
Total videos: 13320

Example classes:
HorseRace: 124 videos
StillRings: 112 videos
ApplyLipstick: 114 videos
HammerThrow: 150 videos
VolleyballSpiking: 116 videos
Biking: 134 videos
PlayingCello: 164 videos
BodyWeightSquats: 112 videos
TaiChi: 100 videos
Punch: 160 videos


## 4. Select balanced classes and videos

In [5]:
REQUESTED = ["Basketball", "Biking", "WalkingWithDog", "Running", "TennisSwing"]

TARGET_CLASSES = 5
MAX_PER_CLASS = 30

available = [c for c in REQUESTED if c in VIDEO_INDEX]

if len(available) < 3:
    raise ValueError(
        f"Only {len(available)} requested classes were found: {available}. "
        "Need at least 3 UCF101 classes."
    )

selected = sorted(available[:TARGET_CLASSES])

n_per = min(
    MAX_PER_CLASS,
    min(len(VIDEO_INDEX[c]) for c in selected)
)

rng = np.random.RandomState(SEED)

paths, labels = [], []

for i, c in enumerate(selected):
    pool = sorted(VIDEO_INDEX[c])
    pick = rng.choice(len(pool), n_per, replace=False)

    for j in pick:
        paths.append(pool[j])
        labels.append(i)

paths = np.array(paths)
labels = np.array(labels)

CLASS_NAMES = selected
NUM_CLASSES = len(CLASS_NAMES)

print("Selected classes :", CLASS_NAMES)
print("Videos per class :", n_per)
print("Total videos     :", len(paths))

for i, c in enumerate(CLASS_NAMES):
    print(f"{c:20s}: {np.sum(labels == i)} videos")

Selected classes : ['Basketball', 'Biking', 'TennisSwing', 'WalkingWithDog']
Videos per class : 30
Total videos     : 120
Basketball          : 30 videos
Biking              : 30 videos
TennisSwing         : 30 videos
WalkingWithDog      : 30 videos


## 5. Video-level train / validation / test split (no leakage)

In [6]:
# Split is on unique video file paths, so one video can appear in exactly one split.
tr_p, tmp_p, tr_y, tmp_y = train_test_split(
    paths, labels, train_size=TRAIN_FRAC, stratify=labels, random_state=SEED)
rel_val = VAL_FRAC / (VAL_FRAC + TEST_FRAC)
va_p, te_p, va_y, te_y = train_test_split(
    tmp_p, tmp_y, train_size=rel_val, stratify=tmp_y, random_state=SEED)

assert len(set(tr_p) & set(va_p)) == 0 and len(set(tr_p) & set(te_p)) == 0 and len(set(va_p) & set(te_p)) == 0
print("Split (videos) -> train:", len(tr_p), "val:", len(va_p), "test:", len(te_p))
for name, y in [("train", tr_y), ("val", va_y), ("test", te_y)]:
    print("  ", name, dict(zip(CLASS_NAMES, np.bincount(y, minlength=NUM_CLASSES))))
print("Leakage check: no video shared between splits -> PASSED")

Split (videos) -> train: 84 val: 18 test: 18
   train {'Basketball': np.int64(21), 'Biking': np.int64(21), 'TennisSwing': np.int64(21), 'WalkingWithDog': np.int64(21)}
   val {'Basketball': np.int64(4), 'Biking': np.int64(5), 'TennisSwing': np.int64(4), 'WalkingWithDog': np.int64(5)}
   test {'Basketball': np.int64(5), 'Biking': np.int64(4), 'TennisSwing': np.int64(5), 'WalkingWithDog': np.int64(4)}
Leakage check: no video shared between splits -> PASSED


## 6. Uniform 10-frame extraction

In [7]:
def sample_frames(path, n=NUM_FRAMES, size=IMG_SIZE):
    # Returns (n, size, size, 3) RGB uint8 array of n UNIFORMLY spaced frames, or None if unreadable.
    cap = cv2.VideoCapture(path)
    if not cap.isOpened(): return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    frames = []
    if total > 0:
        idxs = np.linspace(0, total - 1, n).round().astype(int)
        for i in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
            ok, f = cap.read()
            if not ok: break
            frames.append(cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2RGB), (size, size)))
    if len(frames) < n:                       # fallback: decode sequentially, then subsample uniformly
        cap.release(); cap = cv2.VideoCapture(path); allf = []
        while True:
            ok, f = cap.read()
            if not ok: break
            allf.append(f)
        if len(allf) == 0:
            cap.release(); return None
        idxs = np.linspace(0, len(allf) - 1, n).round().astype(int)
        frames = [cv2.resize(cv2.cvtColor(allf[i], cv2.COLOR_BGR2RGB), (size, size)) for i in idxs]
    cap.release()
    return np.stack(frames[:n]).astype(np.uint8)

# representative video for Plot 01
demo_idx = int(np.where(tr_y == 0)[0][0]); demo_path = tr_p[demo_idx]
demo_frames = sample_frames(demo_path)
print("Representative video:", os.path.basename(demo_path),
      "| class:", CLASS_NAMES[tr_y[demo_idx]], "| frames:", None if demo_frames is None else demo_frames.shape)

Representative video: v_Basketball_g05_c04.avi | class: Basketball | frames: (10, 224, 224, 3)


## 7. Plot 01 — sampled frames

In [8]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(demo_frames[k]); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title("Frame " + str(k + 1), fontsize=12, fontweight="bold")
fig.suptitle("10 Uniformly Sampled Frames  |  Action Class: " + CLASS_NAMES[tr_y[demo_idx]] +
             "  |  " + os.path.basename(demo_path), fontsize=15, fontweight="bold")
GENERATED_PNGS.append(save_fig(fig, "plots", "01_video_sample_frames.png"))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(CLASS_NAMES, [n_per] * NUM_CLASSES, color="#3E7CB1", edgecolor="black")
ax.set_xlabel("Action Class", fontweight="bold"); ax.set_ylabel("Number of Videos", fontweight="bold")
ax.set_title("Class Distribution of Selected UCF101 Subset", fontweight="bold")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
GENERATED_PNGS.append(save_fig(fig, "plots", "09_class_distribution.png"))

saved: experiment6_part2_outputs/plots/01_video_sample_frames.png
saved: experiment6_part2_outputs/plots/09_class_distribution.png


## 8. Frozen MobileNetV2 feature extraction (done ONCE, cached)

In [9]:
cnn = MobileNetV2(weights="imagenet", include_top=False, pooling="avg",
                  input_shape=(IMG_SIZE, IMG_SIZE, 3))
cnn.trainable = False                      # CNN is never trained / fine-tuned
D = int(cnn.output_shape[-1])              # read the dimension programmatically
print("MobileNetV2 loaded | trainable:", cnn.trainable, "| CNN feature dimension D =", D)

def extract_features(split_paths, split_labels, tag):
    X, y, kept, bad = [], [], [], []
    for i, (p, lab) in enumerate(zip(split_paths, split_labels)):
        fr = sample_frames(p)
        if fr is None or fr.shape[0] != NUM_FRAMES:
            bad.append(p); continue
        feats = cnn(preprocess_input(fr.astype("float32")), training=False).numpy()  # (10, D)
        X.append(feats); y.append(lab); kept.append(p)
        if (i + 1) % 20 == 0: print("  " + tag + ": " + str(i + 1) + "/" + str(len(split_paths)))
    if bad: print("  skipped", len(bad), "unreadable video(s) in", tag)
    return np.asarray(X, dtype="float32"), np.asarray(y, dtype="int32"), kept, bad

t0 = time.time()
Xtr, ytr, tr_kept, bad_tr = extract_features(tr_p, tr_y, "train")
Xva, yva, va_kept, bad_va = extract_features(va_p, va_y, "val")
Xte, yte, te_kept, bad_te = extract_features(te_p, te_y, "test")
FEAT_TIME = time.time() - t0
BAD_TOTAL = len(bad_tr) + len(bad_va) + len(bad_te)
print("Feature extraction finished in " + str(round(FEAT_TIME, 1)) + " s | unreadable videos:", BAD_TOTAL)

np.savez_compressed(os.path.join(DIRS["features"], "ucf101_mobilenetv2_features.npz"),
                    Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=yte,
                    classes=np.array(CLASS_NAMES), D=D, num_frames=NUM_FRAMES)
with open(os.path.join(DIRS["features"], "feature_meta.json"), "w") as f:
    json.dump({"classes": CLASS_NAMES, "D": D, "num_frames": NUM_FRAMES,
               "videos_per_class": int(n_per), "seed": SEED,
               "train": len(ytr), "val": len(yva), "test": len(yte),
               "unreadable_videos": BAD_TOTAL, "extraction_seconds": round(FEAT_TIME, 2)}, f, indent=2)
gc.collect()

I0000 00:00:1789914473.438332      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789914473.441581      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
MobileNetV2 loaded | trainable: False | CNN feature dimension D = 1280
  train: 20/84
  train: 40/84
  train: 60/84
  train: 80/84
Feature extraction finished in 26.4 s | unreadable videos: 0


63

## 9. Feature shapes

In [10]:
one_video_shape = Xtr.shape[1:]
full = np.concatenate([Xtr, Xva, Xte], axis=0)
print("Selected classes            :", CLASS_NAMES)
print("Videos per class (selected) :", n_per)
print("CNN feature dimension D     :", D)
print("One-video feature shape     :", tuple(one_video_shape), " (10 x D)")
print("Full feature tensor shape   :", full.shape)
print("Train features              :", Xtr.shape, "labels", ytr.shape)
print("Validation features         :", Xva.shape, "labels", yva.shape)
print("Test features               :", Xte.shape, "labels", yte.shape)
print("Recurrent network input     : (batch, " + str(NUM_FRAMES) + ", " + str(D) + ")")
del full; gc.collect()

pd.DataFrame({
    "Class": CLASS_NAMES,
    "Total Videos": [int(np.sum(np.concatenate([ytr, yva, yte]) == i)) for i in range(NUM_CLASSES)],
    "Train": [int((ytr == i).sum()) for i in range(NUM_CLASSES)],
    "Validation": [int((yva == i).sum()) for i in range(NUM_CLASSES)],
    "Test": [int((yte == i).sum()) for i in range(NUM_CLASSES)],
}).to_csv(os.path.join(DIRS["tables"], "01_dataset_summary.csv"), index=False)

pd.DataFrame([
    ["CNN feature dimension D", str(D)],
    ["Frames per video", str(NUM_FRAMES)],
    ["Frame size", str(IMG_SIZE) + "x" + str(IMG_SIZE) + "x3"],
    ["One video feature shape", str(tuple(one_video_shape))],
    ["Train feature tensor", str(Xtr.shape)],
    ["Validation feature tensor", str(Xva.shape)],
    ["Test feature tensor", str(Xte.shape)],
    ["Recurrent input shape", "(batch, " + str(NUM_FRAMES) + ", " + str(D) + ")"],
    ["Unreadable videos skipped", str(BAD_TOTAL)],
], columns=["Item", "Value"]).to_csv(os.path.join(DIRS["tables"], "02_feature_shapes.csv"), index=False)

Selected classes            : ['Basketball', 'Biking', 'TennisSwing', 'WalkingWithDog']
Videos per class (selected) : 30
CNN feature dimension D     : 1280
One-video feature shape     : (10, 1280)  (10 x D)
Full feature tensor shape   : (120, 10, 1280)
Train features              : (84, 10, 1280) labels (84,)
Validation features         : (18, 10, 1280) labels (18,)
Test features               : (18, 10, 1280) labels (18,)
Recurrent network input     : (batch, 10, 1280)


## 10. Build and train CNN-LSTM and CNN-GRU (identical settings, identical cached features)

In [11]:
def build_model(kind):
    rec = layers.LSTM(RNN_UNITS, name="lstm") if kind == "lstm" else layers.GRU(RNN_UNITS, name="gru")
    m = keras.Sequential([
        layers.Input(shape=(NUM_FRAMES, D)),
        rec,
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ], name="CNN_" + kind.upper())
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=LR),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def train_model(kind):
    tf.keras.utils.set_random_seed(SEED)
    m = build_model(kind)
    with open(os.path.join(DIRS["model_results"], "cnn_" + kind + "_model_summary.txt"), "w") as f:
        m.summary(print_fn=lambda s: f.write(s + "\n"))
    m.summary()
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
    t0 = time.time()
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=MAX_EPOCHS,
              batch_size=BATCH_SIZE, callbacks=[es], verbose=2, shuffle=True)
    tt = time.time() - t0
    params = int(sum(np.prod(w.shape) for w in m.trainable_weights))
    hist = pd.DataFrame(h.history); hist.insert(0, "epoch", np.arange(1, len(hist) + 1))
    hist.to_csv(os.path.join(DIRS["model_results"], "cnn_" + kind + "_training_history.csv"), index=False)
    print(kind.upper() + " trained in " + str(round(tt, 2)) + " s | epochs run: " + str(len(hist)) +
          " | trainable params: " + str(params))
    return m, hist, tt, params

lstm_model, lstm_hist, lstm_time, lstm_params = train_model("lstm")

Model: "CNN_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │       168,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 168,660 (658.83 KB)

 Trainable params: 168,660 (658.83 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
6/6 - 4s - 682ms/step - accuracy: 0.3810 - loss: 1.2710 - val_accuracy: 0.3889 - val_loss: 1.0396
Epoch 2/30
6/6 - 0s - 24ms/step - accuracy: 0.6190 - loss: 0.9307 - val_accuracy: 0.4444 - val_loss: 0.8666
Epoch 3/30
6/6 - 0s - 23ms/step - accuracy: 0.6071 - loss: 0.7883 - val_accuracy: 0.7778 - val_loss: 0.6437
Epoch 4/30
6/6 - 0s - 23ms/step - accuracy: 0.7738 - loss: 0.6547 - val_accuracy: 0.8333 - val_loss: 0.6029
Epoch 5/30
6/6 - 0s - 24ms/step - accuracy: 0.8571 - loss: 0.5542 - val_accuracy: 0.8333 - val_loss: 0.4814
Epoch 6/30
6/6 - 0s - 23ms/step - accuracy: 0.9167 - loss: 0.4611 - val_accuracy: 0.8889 - val_loss: 0.4358
Epoch 7/30
6/6 - 0s - 23ms/step - accuracy: 0.9405 - loss: 0.4116 - val_accuracy: 0.9444 - val_loss: 0.4115
Epoch 8/30
6/6 - 0s - 24ms/step - accuracy: 0.9762 - loss: 0.3614 - val_accuracy: 0.9444 - val_loss: 0.3642
Epoch 9/30
6/6 - 0s - 23ms/step - accuracy: 0.9643 - loss: 0.3062 - val_accuracy: 0.8889 - val_loss: 0.3844
Epoch 10/30
6/6 - 0s - 22ms

In [12]:
# SAME cached CNN features, same labels, same split, same batch size / optimizer / LR / max epochs.
# MobileNetV2 is NOT run again here.
gru_model, gru_hist, gru_time, gru_params = train_model("gru")

Model: "CNN_GRU"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 32)             │       126,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 126,740 (495.08 KB)

 Trainable params: 126,740 (495.08 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
6/6 - 2s - 251ms/step - accuracy: 0.3571 - loss: 1.2981 - val_accuracy: 0.5556 - val_loss: 1.1432
Epoch 2/30
6/6 - 0s - 24ms/step - accuracy: 0.4286 - loss: 1.0823 - val_accuracy: 0.7222 - val_loss: 0.8431
Epoch 3/30
6/6 - 0s - 23ms/step - accuracy: 0.6905 - loss: 0.7527 - val_accuracy: 0.8889 - val_loss: 0.6675
Epoch 4/30
6/6 - 0s - 23ms/step - accuracy: 0.8690 - loss: 0.6200 - val_accuracy: 0.8889 - val_loss: 0.5501
Epoch 5/30
6/6 - 0s - 23ms/step - accuracy: 0.9524 - loss: 0.4982 - val_accuracy: 0.8333 - val_loss: 0.5178
Epoch 6/30
6/6 - 0s - 23ms/step - accuracy: 0.9405 - loss: 0.4209 - val_accuracy: 0.8889 - val_loss: 0.4442
Epoch 7/30
6/6 - 0s - 23ms/step - accuracy: 1.0000 - loss: 0.3588 - val_accuracy: 0.8889 - val_loss: 0.4051
Epoch 8/30
6/6 - 0s - 23ms/step - accuracy: 1.0000 - loss: 0.2971 - val_accuracy: 0.8889 - val_loss: 0.3827
Epoch 9/30
6/6 - 0s - 23ms/step - accuracy: 0.9881 - loss: 0.2555 - val_accuracy: 0.8889 - val_loss: 0.3571
Epoch 10/30
6/6 - 0s - 23ms

## 11. Training curves (Plots 02-05)

In [13]:
def curve(hist, key, ylabel, title, fname):
    fig, ax = plt.subplots(figsize=(8, 5))
    ep = hist["epoch"]
    ys, yv = (hist[key] * 100, hist["val_" + key] * 100) if key == "accuracy" else (hist[key], hist["val_" + key])
    ax.plot(ep, ys, "o-", color="#1f77b4", lw=2, ms=4, label="Training")
    ax.plot(ep, yv, "s--", color="#d62728", lw=2, ms=4, label="Validation")
    ax.set_xlabel("Epoch", fontweight="bold"); ax.set_ylabel(ylabel, fontweight="bold")
    ax.set_title(title, fontweight="bold"); ax.legend(frameon=True)
    return save_fig(fig, "plots", fname)

GENERATED_PNGS += [
    curve(lstm_hist, "loss", "Loss", "CNN-LSTM: Training vs Validation Loss", "02_cnn_lstm_training_loss.png"),
    curve(lstm_hist, "accuracy", "Accuracy (%)", "CNN-LSTM: Training vs Validation Accuracy", "03_cnn_lstm_training_accuracy.png"),
    curve(gru_hist, "loss", "Loss", "CNN-GRU: Training vs Validation Loss", "04_cnn_gru_training_loss.png"),
    curve(gru_hist, "accuracy", "Accuracy (%)", "CNN-GRU: Training vs Validation Accuracy", "05_cnn_gru_training_accuracy.png"),
]

saved: experiment6_part2_outputs/plots/02_cnn_lstm_training_loss.png
saved: experiment6_part2_outputs/plots/03_cnn_lstm_training_accuracy.png
saved: experiment6_part2_outputs/plots/04_cnn_gru_training_loss.png
saved: experiment6_part2_outputs/plots/05_cnn_gru_training_accuracy.png


## 12. Test evaluation

In [14]:
def evaluate(model, name, kind):
    probs = model.predict(Xte, verbose=0)
    pred = probs.argmax(1)
    res = {"Model": name,
           "Accuracy (%)": accuracy_score(yte, pred) * 100,
           "Macro Precision (%)": precision_score(yte, pred, average="macro", zero_division=0) * 100,
           "Macro Recall (%)": recall_score(yte, pred, average="macro", zero_division=0) * 100,
           "Macro F1 (%)": f1_score(yte, pred, average="macro", zero_division=0) * 100}
    rep = classification_report(yte, pred, target_names=CLASS_NAMES, digits=4, zero_division=0)
    with open(os.path.join(DIRS["model_results"], "cnn_" + kind + "_classification_report.txt"), "w") as f:
        f.write(name + " - test classification report\n\n" + rep)
    print("\n===== " + name + " =====")
    for k, v in res.items():
        print("  " + k + ": " + (v if isinstance(v, str) else str(round(v, 2))))
    print(rep)
    return res, probs, pred

lstm_res, lstm_probs, lstm_pred = evaluate(lstm_model, "CNN-LSTM", "lstm")
gru_res,  gru_probs,  gru_pred  = evaluate(gru_model,  "CNN-GRU",  "gru")


===== CNN-LSTM =====
  Model: CNN-LSTM
  Accuracy (%): 100.0
  Macro Precision (%): 100.0
  Macro Recall (%): 100.0
  Macro F1 (%): 100.0
                precision    recall  f1-score   support

    Basketball     1.0000    1.0000    1.0000         5
        Biking     1.0000    1.0000    1.0000         4
   TennisSwing     1.0000    1.0000    1.0000         5
WalkingWithDog     1.0000    1.0000    1.0000         4

      accuracy                         1.0000        18
     macro avg     1.0000    1.0000    1.0000        18
  weighted avg     1.0000    1.0000    1.0000        18


===== CNN-GRU =====
  Model: CNN-GRU
  Accuracy (%): 100.0
  Macro Precision (%): 100.0
  Macro Recall (%): 100.0
  Macro F1 (%): 100.0
                precision    recall  f1-score   support

    Basketball     1.0000    1.0000    1.0000         5
        Biking     1.0000    1.0000    1.0000         4
   TennisSwing     1.0000    1.0000    1.0000         5
WalkingWithDog     1.0000    1.0000    1.0000   

## 13. Confusion matrices (Plots 07-08)

In [15]:
def plot_cm(pred, title, fname):
    cm = confusion_matrix(yte, pred, labels=range(NUM_CLASSES))
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow(cm, cmap="Blues"); ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.85, label="Number of Videos")
    thr = cm.max() / 2 if cm.max() else 0.5
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=13, fontweight="bold",
                    color="white" if cm[i, j] > thr else "black")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right"); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted Class", fontweight="bold"); ax.set_ylabel("Actual Class", fontweight="bold")
    ax.set_title(title, fontweight="bold")
    return save_fig(fig, "confusion_matrices", fname), cm

p1, cm_lstm = plot_cm(lstm_pred, "CNN-LSTM Confusion Matrix (Test Set)", "07_cnn_lstm_confusion_matrix.png")
p2, cm_gru  = plot_cm(gru_pred,  "CNN-GRU Confusion Matrix (Test Set)",  "08_cnn_gru_confusion_matrix.png")
GENERATED_PNGS += [p1, p2]
print("CNN-LSTM confusion matrix:\n", cm_lstm)
print("CNN-GRU confusion matrix:\n", cm_gru)

saved: experiment6_part2_outputs/confusion_matrices/07_cnn_lstm_confusion_matrix.png
saved: experiment6_part2_outputs/confusion_matrices/08_cnn_gru_confusion_matrix.png
CNN-LSTM confusion matrix:
 [[5 0 0 0]
 [0 4 0 0]
 [0 0 5 0]
 [0 0 0 4]]
CNN-GRU confusion matrix:
 [[5 0 0 0]
 [0 4 0 0]
 [0 0 5 0]
 [0 0 0 4]]


## 14. Test prediction with model confidence

In [16]:
rows = []
for i in range(len(yte)):
    rows.append({"Video": os.path.basename(te_kept[i]), "Actual Class": CLASS_NAMES[yte[i]],
                 "LSTM Predicted": CLASS_NAMES[lstm_pred[i]], "LSTM Confidence": float(lstm_probs[i].max()),
                 "GRU Predicted": CLASS_NAMES[gru_pred[i]],  "GRU Confidence": float(gru_probs[i].max())})
pred_df = pd.DataFrame(rows)
pred_df.to_csv(os.path.join(DIRS["tables"], "04_test_predictions.csv"), index=False)

k = 0  # example test video
EX = {"video": os.path.basename(te_kept[k]), "actual": CLASS_NAMES[yte[k]],
      "lstm_pred": CLASS_NAMES[lstm_pred[k]], "lstm_conf": float(lstm_probs[k].max()),
      "gru_pred": CLASS_NAMES[gru_pred[k]],   "gru_conf": float(gru_probs[k].max())}
print("Example test video:", EX["video"])
print("CNN-LSTM  -> Predicted class : " + EX["lstm_pred"])
print("             Actual class    : " + EX["actual"])
print("             Confidence      : " + str(round(EX["lstm_conf"], 4)) + "   (from model.predict())")
print("CNN-GRU   -> Predicted class : " + EX["gru_pred"])
print("             Actual class    : " + EX["actual"])
print("             Confidence      : " + str(round(EX["gru_conf"], 4)) + "   (from model.predict())")
pred_df.head(10)

Example test video: v_WalkingWithDog_g15_c02.avi
CNN-LSTM  -> Predicted class : WalkingWithDog
             Actual class    : WalkingWithDog
             Confidence      : 0.992   (from model.predict())
CNN-GRU   -> Predicted class : WalkingWithDog
             Actual class    : WalkingWithDog
             Confidence      : 0.9671   (from model.predict())


,Video,Actual Class,LSTM Predicted,LSTM Confidence,GRU Predicted,GRU Confidence
0,v_WalkingWithDog_g15_c02.avi,WalkingWithDog,WalkingWithDog,0.991989,WalkingWithDog,0.967111
1,v_TennisSwing_g09_c02.avi,TennisSwing,TennisSwing,0.984519,TennisSwing,0.982382
2,v_Biking_g02_c04.avi,Biking,Biking,0.960216,Biking,0.967027
3,v_Basketball_g19_c04.avi,Basketball,Basketball,0.987370,Basketball,0.990062
4,v_TennisSwing_g16_c03.avi,TennisSwing,TennisSwing,0.977587,TennisSwing,0.968693
5,v_TennisSwing_g16_c04.avi,TennisSwing,TennisSwing,0.983615,TennisSwing,0.978475
6,v_WalkingWithDog_g11_c01.avi,WalkingWithDog,WalkingWithDog,0.991978,WalkingWithDog,0.986358
7,v_WalkingWithDog_g05_c01.avi,WalkingWithDog,WalkingWithDog,0.956641,WalkingWithDog,0.973902
8,v_Basketball_g20_c01.avi,Basketball,Basketball,0.981761,Basketball,0.983224
9,v_Basketball_g14_c02.avi,Basketball,Basketball,0.979637,Basketball,0.931042


## 15. LSTM vs GRU comparison (Additional Exercise #6) — Plot 06 + table

In [17]:
comp = pd.DataFrame([
    {**lstm_res, "Parameters": lstm_params, "Training Time (s)": round(lstm_time, 2)},
    {**gru_res,  "Parameters": gru_params,  "Training Time (s)": round(gru_time, 2)},
])[["Model", "Accuracy (%)", "Macro Precision (%)", "Macro Recall (%)", "Macro F1 (%)",
    "Parameters", "Training Time (s)"]]
for c in ["Accuracy (%)", "Macro Precision (%)", "Macro Recall (%)", "Macro F1 (%)"]:
    comp[c] = comp[c].round(2)
comp.to_csv(os.path.join(DIRS["tables"], "03_video_model_comparison.csv"), index=False)
print(comp.to_string(index=False))

models = comp["Model"].tolist(); x = np.arange(2); w = 0.35
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5.5))
b1 = a1.bar(x - w/2, comp["Accuracy (%)"], w, label="Test Accuracy (%)", color="#3E7CB1", edgecolor="black")
b2 = a1.bar(x + w/2, comp["Macro F1 (%)"], w, label="Test Macro F1 (%)", color="#E1A140", edgecolor="black")
for b in list(b1) + list(b2):
    a1.text(b.get_x() + b.get_width()/2, b.get_height() + 1, str(round(b.get_height(), 2)),
            ha="center", fontsize=11, fontweight="bold")
a1.set_xticks(x); a1.set_xticklabels(models); a1.set_ylim(0, 110)
a1.set_xlabel("Model", fontweight="bold"); a1.set_ylabel("Score (%)", fontweight="bold")
a1.set_title("Predictive Performance (Test Set)", fontweight="bold"); a1.legend()

b3 = a2.bar(x - w/2, comp["Parameters"] / 1000.0, w, label="Trainable Parameters (thousands)",
            color="#6A994E", edgecolor="black")
b4 = a2.bar(x + w/2, comp["Training Time (s)"], w, label="Training Time (s)",
            color="#BC4749", edgecolor="black")
for b in list(b3) + list(b4):
    a2.text(b.get_x() + b.get_width()/2, b.get_height(), str(round(b.get_height(), 2)),
            ha="center", va="bottom", fontsize=11, fontweight="bold")
a2.set_xticks(x); a2.set_xticklabels(models)
a2.set_xlabel("Model", fontweight="bold"); a2.set_ylabel("Parameters (k)  /  Time (s)", fontweight="bold")
a2.set_title("Model Complexity and Training Cost", fontweight="bold"); a2.legend()
fig.suptitle("CNN-LSTM vs CNN-GRU on Identical Cached MobileNetV2 Features", fontsize=16, fontweight="bold")
GENERATED_PNGS.append(save_fig(fig, "plots", "06_video_model_comparison.png"))

   Model  Accuracy (%)  Macro Precision (%)  Macro Recall (%)  Macro F1 (%)  Parameters  Training Time (s)
CNN-LSTM         100.0                100.0             100.0         100.0      168660               8.15
 CNN-GRU         100.0                100.0             100.0         100.0      126740               5.56
saved: experiment6_part2_outputs/plots/06_video_model_comparison.png


## 16. Auto-generated factual report text

In [18]:
def curve_note(h, tag):
    last = h.iloc[-1]; best = h["val_loss"].idxmin() + 1
    gap = (last["accuracy"] - last["val_accuracy"]) * 100
    return (tag + ": epochs run = " + str(len(h)) + ", best val_loss at epoch " + str(int(best)) +
            " (" + str(round(h['val_loss'].min(), 4)) + "), final train/val loss = " +
            str(round(last['loss'], 4)) + "/" + str(round(last['val_loss'], 4)) +
            ", final train/val accuracy = " + str(round(last['accuracy'] * 100, 2)) + "%/" +
            str(round(last['val_accuracy'] * 100, 2)) + "%, train-val accuracy gap = " +
            str(round(gap, 2)) + " points.")

def cm_note(cm, tag):
    diag = np.diag(cm); support = cm.sum(1)
    rec = np.divide(diag, np.maximum(support, 1))
    best_i, worst_i = int(rec.argmax()), int(rec.argmin())
    off = cm.copy(); np.fill_diagonal(off, 0)
    lines = [tag + ": highest per-class recall = " + CLASS_NAMES[best_i] + " (" +
             str(diag[best_i]) + "/" + str(support[best_i]) + "), lowest = " + CLASS_NAMES[worst_i] +
             " (" + str(diag[worst_i]) + "/" + str(support[worst_i]) + "), total misclassified = " +
             str(int(off.sum())) + "/" + str(int(cm.sum())) + "."]
    if off.sum() > 0:
        i, j = np.unravel_index(off.argmax(), off.shape)
        lines.append(tag + ": most frequent confusion = actual " + CLASS_NAMES[i] + " predicted as " +
                     CLASS_NAMES[j] + " (" + str(int(off[i, j])) + " video(s)).")
    else:
        lines.append(tag + ": no off-diagonal entries; all test videos classified correctly.")
    return "\n".join(lines)

L = []
L.append("EXPERIMENT 6 - PART 2: VIDEO UNDERSTANDING USING CNN + RNN (auto-generated from this run)")
L.append("")
L.append("DATASET")
L.append("Dataset: UCF101, located at " + str(UCF_ROOT) + ".")
L.append("Selected classes (" + str(NUM_CLASSES) + "): " + ", ".join(CLASS_NAMES) + ".")
L.append("Videos per class: " + str(n_per) + "; total videos used: " + str(len(ytr) + len(yva) + len(yte)) + ".")
L.append("Video-level split (70/15/15, seed " + str(SEED) + "): train " + str(len(ytr)) +
         ", validation " + str(len(yva)) + ", test " + str(len(yte)) + ". No video appears in more than one split.")
L.append("Unreadable/invalid videos skipped: " + str(BAD_TOTAL) + ".")
L.append("")
L.append("CNN FEATURE EXTRACTION")
L.append("Feature extractor: MobileNetV2 (weights=imagenet, include_top=False, pooling=avg), frozen (trainable=False).")
L.append("Frames per video: " + str(NUM_FRAMES) + ", uniformly sampled across the whole video, resized to " +
         str(IMG_SIZE) + "x" + str(IMG_SIZE) + "x3 RGB.")
L.append("CNN feature dimension D = " + str(D) + ".")
L.append("One-video feature shape: " + str(tuple(one_video_shape)) + ".")
L.append("Recurrent input shape: (batch, " + str(NUM_FRAMES) + ", " + str(D) + ").")
L.append("Feature tensors: train " + str(Xtr.shape) + ", validation " + str(Xva.shape) + ", test " + str(Xte.shape) + ".")
L.append("Feature extraction time: " + str(round(FEAT_TIME, 2)) + " s; features cached once and reused by both models.")
L.append("")
L.append("TRAINING CONFIGURATION (identical for both models)")
L.append("Optimizer Adam, learning rate " + str(LR) + ", loss sparse categorical crossentropy, batch size " +
         str(BATCH_SIZE) + ", maximum " + str(MAX_EPOCHS) + " epochs, recurrent units " + str(RNN_UNITS) +
         ", early stopping on val_loss (patience 8, best weights restored).")
L.append("")
L.append("TEST METRICS")
for r in comp.to_dict("records"):
    L.append(r["Model"] + ": accuracy " + str(r["Accuracy (%)"]) + "%, macro precision " +
             str(r["Macro Precision (%)"]) + "%, macro recall " + str(r["Macro Recall (%)"]) +
             "%, macro F1 " + str(r["Macro F1 (%)"]) + "%, trainable parameters " + str(r["Parameters"]) +
             ", training time " + str(r["Training Time (s)"]) + " s.")
L.append("Parameter difference (LSTM - GRU): " + str(lstm_params - gru_params) + ".")
L.append("Training-time difference (LSTM - GRU): " + str(round(lstm_time - gru_time, 2)) + " s.")
L.append("Macro F1 difference (LSTM - GRU): " + str(round(lstm_res["Macro F1 (%)"] - gru_res["Macro F1 (%)"], 2)) + " points.")
L.append("")
L.append("EXAMPLE TEST PREDICTION (confidence taken directly from model.predict())")
L.append("Video: " + EX["video"])
L.append("CNN-LSTM predicted class: " + EX["lstm_pred"] + " | actual class: " + EX["actual"] +
         " | confidence: " + str(round(EX["lstm_conf"], 4)))
L.append("CNN-GRU predicted class: " + EX["gru_pred"] + " | actual class: " + EX["actual"] +
         " | confidence: " + str(round(EX["gru_conf"], 4)))
L.append("")
L.append("TRAINING-CURVE OBSERVATIONS")
L.append(curve_note(lstm_hist, "CNN-LSTM")); L.append(curve_note(gru_hist, "CNN-GRU"))
L.append("")
L.append("CONFUSION-MATRIX OBSERVATIONS")
L.append(cm_note(cm_lstm, "CNN-LSTM")); L.append(cm_note(cm_gru, "CNN-GRU"))
L.append("")
L.append("CNN SPATIAL VS RECURRENT TEMPORAL INFORMATION")
L.append("The frozen MobileNetV2 encodes per-frame spatial appearance (objects, pose, scene layout, texture) into a " +
         str(D) + "-dimensional vector; it is applied independently to each of the " + str(NUM_FRAMES) +
         " frames and carries no information about frame order.")
L.append("The recurrent layer receives the ordered sequence of " + str(NUM_FRAMES) +
         " spatial vectors and models how appearance evolves over time, so the action label is decided from motion " +
         "and temporal context rather than from any single frame.")
L.append("Because both recurrent models consume byte-identical cached features, the same split and the same " +
         "hyperparameters, the metric differences above are attributable only to the LSTM vs GRU gating mechanism.")

txt = "\n".join(L)
with open(os.path.join(DIRS["model_results"], "part2_auto_generated_report_text.txt"), "w") as f:
    f.write(txt)
print(txt)

EXPERIMENT 6 - PART 2: VIDEO UNDERSTANDING USING CNN + RNN (auto-generated from this run)

DATASET
Dataset: UCF101, located at /kaggle/input/datasets/pevogam/ucf101/UCF101/UCF-101.
Selected classes (4): Basketball, Biking, TennisSwing, WalkingWithDog.
Videos per class: 30; total videos used: 120.
Video-level split (70/15/15, seed 42): train 84, validation 18, test 18. No video appears in more than one split.
Unreadable/invalid videos skipped: 0.

CNN FEATURE EXTRACTION
Feature extractor: MobileNetV2 (weights=imagenet, include_top=False, pooling=avg), frozen (trainable=False).
Frames per video: 10, uniformly sampled across the whole video, resized to 224x224x3 RGB.
CNN feature dimension D = 1280.
One-video feature shape: (10, 1280).
Recurrent input shape: (batch, 10, 1280).
Feature tensors: train (84, 10, 1280), validation (18, 10, 1280), test (18, 10, 1280).
Feature extraction time: 26.36 s; features cached once and reused by both models.

TRAINING CONFIGURATION (identical for both mod

## 17. Final summary

In [19]:
print("=" * 78)
print("EXPERIMENT 6 - PART 2 + ADDITIONAL EXERCISE #6 : FINAL SUMMARY")
print("=" * 78)
print("Selected classes        :", CLASS_NAMES)
print("Videos per class        :", n_per, "| train/val/test:", len(ytr), "/", len(yva), "/", len(yte))
print("CNN feature dimension D :", D)
print("One-video feature shape :", tuple(one_video_shape))
print("Feature tensors         : train", Xtr.shape, "| val", Xva.shape, "| test", Xte.shape)
print("Recurrent input shape   : (batch,", NUM_FRAMES, ",", D, ")")
print("-" * 78)
print(comp.to_string(index=False))
print("-" * 78)
print("Trainable parameters    : CNN-LSTM", lstm_params, "| CNN-GRU", gru_params)
print("Training time (s)       : CNN-LSTM", round(lstm_time, 2), "| CNN-GRU", round(gru_time, 2))
print("Output directory        :", os.path.abspath(OUT))
print("Generated PNG files:")
for p in GENERATED_PNGS: print("   ", p)
print("Other output files:")
for root, _, files in os.walk(OUT):
    for fn in sorted(files):
        if not fn.endswith(".png"): print("   ", os.path.join(root, fn))
print("=" * 78)

EXPERIMENT 6 - PART 2 + ADDITIONAL EXERCISE #6 : FINAL SUMMARY
Selected classes        : ['Basketball', 'Biking', 'TennisSwing', 'WalkingWithDog']
Videos per class        : 30 | train/val/test: 84 / 18 / 18
CNN feature dimension D : 1280
One-video feature shape : (10, 1280)
Feature tensors         : train (84, 10, 1280) | val (18, 10, 1280) | test (18, 10, 1280)
Recurrent input shape   : (batch, 10 , 1280 )
------------------------------------------------------------------------------
   Model  Accuracy (%)  Macro Precision (%)  Macro Recall (%)  Macro F1 (%)  Parameters  Training Time (s)
CNN-LSTM         100.0                100.0             100.0         100.0      168660               8.15
 CNN-GRU         100.0                100.0             100.0         100.0      126740               5.56
------------------------------------------------------------------------------
Trainable parameters    : CNN-LSTM 168660 | CNN-GRU 126740
Training time (s)       : CNN-LSTM 8.15 | CNN-GRU 5

In [20]:
import shutil

shutil.make_archive(
    "/kaggle/working/experiment6_part2_outputs",
    "zip",
    "/kaggle/working/experiment6_part2_outputs"
)

print("/kaggle/working/experiment6_part2_outputs.zip")

/kaggle/working/experiment6_part2_outputs.zip
